# Perturbation XAI cho hai checkpoint VG/PSG


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, importlib.util, json, statistics, subprocess, sys
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
REPO = DRIVE / 'graphml-project3'
RUNS = REPO / 'duong/runs/qa_vg_psg_xai'
MANIFEST = None  # None = manifest huấn luyện mới nhất
SEEDS = (0, 1, 2)
N_PER_STRATUM = 3
MAX_REL_EDGES = 40
RANDOM_REPEATS = 20
EXPLANATION_SIZES = (1, 3, 5)
SELECTION_SEED = 42
assert (REPO / 'duong/xai/model.py').is_file(), 'Đồng bộ repo lên Drive.'
if importlib.util.find_spec('torch_geometric') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO / 'huy')], check=True)
sys.path[:0] = [str(REPO / 'duong'), str(REPO / 'huy/src')]
import torch
from xai.model import HuyQACaseModel, MODEL_SPECS
from xai.explain import EdgeContributionExplainer
from xai.evaluate_xai import XAIEvaluator
from xai.perturbation import rank_of, score_without_edges
from xai.io import explanation_to_dict
from mmxai.data.task1 import load_task1
from mmxai.graph.build import build_graph
from mmxai.graph.qa import QAData, QAScorer, qa_scores
from mmxai.experiments.t1_graph import load_eval_module
assert torch.cuda.is_available(), 'Chọn GPU trong Colab Runtime.'
device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4


In [2]:
if MANIFEST is None:
    manifests = sorted(RUNS.glob('dual-*_manifest.json'), key=lambda p: p.stat().st_mtime)
    assert manifests, f'Không tìm thấy manifest trong {RUNS}'
    MANIFEST = manifests[-1]
MANIFEST = Path(MANIFEST)
m = json.loads(MANIFEST.read_text())
assert len(m['runs']) == len(MODEL_SPECS) * len(SEEDS)
records = {(r['dataset'], int(r['seed'])): r for r in m['runs']}
assert set(records) == {(name, seed) for name in MODEL_SPECS for seed in SEEDS}
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
for rel, expected in m['source_sha256'].items():
    assert sha256(REPO / rel) == expected, f'Mã nguồn đổi từ khi huấn luyện: {rel}'
for name, spec in MODEL_SPECS.items():
    bundle_dir = DRIVE / spec.bundle_dir
    for rel, expected in m['bundle_sha256'][name].items():
        assert sha256(bundle_dir / rel) == expected, f'{name}: dữ liệu đổi: {rel}'
    for seed in SEEDS:
        rec = records[name, seed]
        checkpoint = Path(rec['run_dir']) / 'model.pt'
        assert checkpoint.is_file() and sha256(checkpoint) == rec['checkpoint_sha256']
OUT = RUNS / datetime.now(timezone.utc).strftime('xai-%Y%m%d-%H%M%S')
OUT.mkdir(parents=True, exist_ok=False)
print('Manifest:', MANIFEST, '| output:', OUT)


Manifest: /content/drive/MyDrive/graphml-project3/duong/runs/qa_vg_psg_xai/dual-20261121-125423_manifest.json | output: /content/drive/MyDrive/graphml-project3/duong/runs/qa_vg_psg_xai/xai-20261121-130842


In [3]:
all_rows, all_explanations, selected_cases = [], [], []
for name, spec in MODEL_SPECS.items():
    bundle_dir = DRIVE / spec.bundle_dir
    ev = load_eval_module(bundle_dir)
    protocol = ev.Protocol.load(bundle_dir)
    bundle = load_task1(bundle_dir, bundle_dir / 'huy_extra_cn')
    cap_concepts = {int(c['caption_id']): c['concepts'] for c in bundle.captions}
    cands = protocol.candidates('test', None)
    seed0 = records[name, 0]
    saved_seed0 = json.loads((Path(seed0['run_dir']) / 'ranks_test_full.json').read_text())['ranks']
    assert len(saved_seed0) == len(cands['q_cap'])
    rng = np.random.default_rng(SELECTION_SEED)
    case_indices = []
    for group in ('direct', 'indirect'):
        for outcome in ('success', 'failure'):
            pool = [j for j, rank in enumerate(saved_seed0)
                    if cands['group'][j] == group and int(cands['pos'][j]) >= 0
                    and ((int(rank) == 1) if outcome == 'success' else (int(rank) > 10))]
            assert len(pool) >= N_PER_STRATUM, f'{name}: thiếu case {group}/{outcome}: {len(pool)}'
            for j in sorted(rng.choice(pool, N_PER_STRATUM, replace=False).tolist()):
                case_indices.append((int(j), group, outcome))
                selected_cases.append({'dataset': name, 'candidate_row': int(j),
                                       'caption_id': int(cands['q_cap'][j]), 'group': group,
                                       'seed0_outcome': outcome,
                                       'ground_truth_image_id': int(cands['q_img'][j])})
    x_img = torch.from_numpy(bundle.image_emb).to(device)
    for seed in SEEDS:
        rec = records[name, seed]
        run_dir = Path(rec['run_dir'])
        cfg = json.loads((run_dir / 'config.json').read_text())
        graph = build_graph(bundle, variant=cfg['graph']['variant'],
                            min_rel_count=int(cfg['graph'].get('min_rel_count', 1)), seed=seed,
                            rel_merge=cfg['graph'].get('rel_merge', 'directed'))
        stats = json.loads((run_dir / 'metrics.json').read_text())['run']['extra']['graph']
        assert graph.stats['n_label'] == stats['n_label']
        assert graph.stats['E_concept'] == stats['E_concept']
        qd = QAData(graph, cap_concepts, device)
        model = QAScorer(bundle.image_emb.shape[1], **cfg['gnn']).to(device)
        model.load_state_dict(torch.load(run_dir / 'model.pt', map_location='cpu', weights_only=True))
        model.eval()
        saved_ranks = json.loads((run_dir / 'ranks_test_full.json').read_text())['ranks']
        assert len(saved_ranks) == len(cands['q_cap'])
        for j, group, outcome in case_indices:
            case = HuyQACaseModel(model, graph, qd, bundle, protocol, cands, ev, j,
                                  rec['alpha'], device, MAX_REL_EDGES, image_tensor=x_img)
            base = case.score_removed()
            reference = qa_scores(model, qd, x_img,
                                  np.asarray(protocol.cap[case.cap_id:case.cap_id+1]),
                                  np.asarray([case.cap_id]), case.rows_np[None], device)[0]
            np.testing.assert_allclose(base['qa'], reference, rtol=1e-4, atol=1e-5)
            query = protocol.captions[case.cap_id]['text']
            candidate_ids = tuple(str(i) for i in case.ids)
            graph_xai, scorer = case.intervention_graph, case.scorer
            before = scorer.score(query, candidate_ids, graph_xai)
            gt_id = str(int(cands['q_img'][j]))
            gt_rank = rank_of(before, gt_id, candidate_ids)
            assert gt_rank == int(saved_ranks[j]), (name, seed, case.cap_id, gt_rank, saved_ranks[j])
            for k in EXPLANATION_SIZES:
                explanation = EdgeContributionExplainer(top_k=k).explain(
                    query, str(case.top_id), candidate_ids, graph_xai, scorer)
                random_seed = SELECTION_SEED + seed*100000 + j*10 + k
                result = XAIEvaluator(case.schema, random_repeats=RANDOM_REPEATS,
                                      seed=random_seed,
                                      match_random_edge_type=True).evaluate(
                    query, str(case.top_id), candidate_ids, graph_xai, scorer, explanation)
                after = score_without_edges(scorer, query, candidate_ids,
                                            graph_xai, explanation.edge_ids).scores
                metrics = result.to_dict()
                metrics['random_seed'] = metrics.pop('seed')
                row = {'dataset': name, 'seed': seed, 'group': group, 'seed0_outcome': outcome,
                       'caption_id': case.cap_id, 'ground_truth_image_id': gt_id,
                       'gt_rank_before': gt_rank, 'gt_rank_after': rank_of(after, gt_id, candidate_ids),
                       'requested_k': k, 'screen_coverage': len(case.screened)/max(1,len(case.local_all)),
                       'local_edges_full': len(case.local_all), 'alpha': rec['alpha'],
                       **metrics}
                all_rows.append(row)
                all_explanations.append({'dataset': name, 'seed': seed, 'caption_id': case.cap_id,
                                         'requested_k': k,
                                         'explanation': explanation_to_dict(explanation, graph_xai)})
        (OUT / f'{name}_seed{seed}.json').write_text(json.dumps(
            [r for r in all_rows if r['dataset']==name and r['seed']==seed],
            ensure_ascii=False, indent=2))
        (OUT / f'{name}_seed{seed}_explanations.json').write_text(json.dumps(
            [r for r in all_explanations if r['dataset']==name and r['seed']==seed],
            ensure_ascii=False, indent=2))
        print('Done:', name, seed, '|', sum(r['dataset']==name and r['seed']==seed for r in all_rows), 'rows')
(OUT / 'selected_cases.json').write_text(json.dumps(selected_cases, ensure_ascii=False, indent=2))
(OUT / 'explanations.json').write_text(json.dumps(all_explanations, ensure_ascii=False, indent=2))


[    0.9s] Protocol: 19264 ảnh, 96381 caption, features clip_ViT-B-32_laion2b.npz (19264, 512), gallery=1000, K=100


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Done: vg 0 | 0 rows
Done: vg 1 | 0 rows
Done: vg 2 | 0 rows
[    1.1s] Protocol: 18186 ảnh, 90974 caption, features clip_ViT-B-32_laion2b.npz (18186, 512), gallery=1000, K=100
Done: psg 0 | 0 rows
Done: psg 1 | 0 rows
Done: psg 2 | 0 rows


1011460

In [4]:
assert len(all_rows) == len(MODEL_SPECS) * len(SEEDS) * 12 * len(EXPLANATION_SIZES)
with (OUT / 'xai_results.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(all_rows[0]))
    writer.writeheader(); writer.writerows(all_rows)
summary = []
for name in MODEL_SPECS:
    for group in ('direct', 'indirect'):
        for outcome in ('success', 'failure'):
            for k in EXPLANATION_SIZES:
                subset = [r for r in all_rows if r['dataset']==name and r['group']==group
                          and r['seed0_outcome']==outcome and r['requested_k']==k]
                summary.append({'dataset': name, 'group': group, 'seed0_outcome': outcome,
                                'k': k, 'n': len(subset),
                                'mean_final_drop': statistics.fmean(r['final_score_drop'] for r in subset),
                                'mean_random_drop': statistics.fmean(r['random_final_score_drop_mean'] for r in subset),
                                'mean_advantage': statistics.fmean(r['fidelity_advantage'] for r in subset),
                                'fraction_top1_changed': statistics.fmean(r['rank_after'] > 1 for r in subset),
                                'mean_coverage': statistics.fmean(r['screen_coverage'] for r in subset)})
(OUT / 'summary.json').write_text(json.dumps({'training_manifest': str(MANIFEST),
    'n_selected_cases': len(selected_cases), 'n_rows': len(all_rows),
    'random_repeats': RANDOM_REPEATS, 'match_random_edge_type': True,
    'max_relation_edges': MAX_REL_EDGES, 'summary': summary}, ensure_ascii=False, indent=2))
for row in summary: print(row)
print('Saved:', OUT)


{'dataset': 'vg', 'group': 'direct', 'seed0_outcome': 'success', 'k': 1, 'n': 9, 'mean_final_drop': 0.5649218288335405, 'mean_random_drop': 0.055493435915917017, 'mean_advantage': 0.5094283929176235, 'fraction_top1_changed': 0.6666666666666666, 'mean_coverage': 0.008837471204736314}
{'dataset': 'vg', 'group': 'direct', 'seed0_outcome': 'success', 'k': 3, 'n': 9, 'mean_final_drop': 1.2696555448361624, 'mean_random_drop': 0.285919761926678, 'mean_advantage': 0.9837357829094844, 'fraction_top1_changed': 0.6666666666666666, 'mean_coverage': 0.008837471204736314}
{'dataset': 'vg', 'group': 'direct', 'seed0_outcome': 'success', 'k': 5, 'n': 9, 'mean_final_drop': 1.4128838297125468, 'mean_random_drop': 0.3522993746737762, 'mean_advantage': 1.0605844550387706, 'fraction_top1_changed': 0.6666666666666666, 'mean_coverage': 0.008837471204736314}
{'dataset': 'vg', 'group': 'direct', 'seed0_outcome': 'failure', 'k': 1, 'n': 9, 'mean_final_drop': 0.7900047976154324, 'mean_random_drop': 0.09502887774